# 🛡️ Errors — the basics

**Time:** ~30 minutes &nbsp;•&nbsp; **Goal:** stop fearing red error messages — **read** them, **catch** them, and **raise** your own.

Your `translate()` works beautifully on clean genes. But real sequence files are **messy**: a base the machine wasn't sure of (`N`), lower-case letters, a fragment that isn't a whole number of codons. Feed those in and Python **crashes**. This notebook is about staying in control when that happens.

Skills:

1. Read a traceback — errors are *information*, not failure
2. `try` / `except` — catch a crash and carry on
3. Catch the **right** error — be specific
4. `raise` your own — reject bad data on purpose

## Our data — clean genes, and some messy ones

Run this first. Alongside the two species you now have deliberately **broken** sequences to practise on.

In [ ]:
# The genetic code: every three-letter codon -> its amino acid (one letter).
codon_table = {
    "TTT": "F", "TTC": "F", "TTA": "L", "TTG": "L",
    "CTT": "L", "CTC": "L", "CTA": "L", "CTG": "L",
    "ATT": "I", "ATC": "I", "ATA": "I", "ATG": "M",
    "GTT": "V", "GTC": "V", "GTA": "V", "GTG": "V",
    "TCT": "S", "TCC": "S", "TCA": "S", "TCG": "S",
    "CCT": "P", "CCC": "P", "CCA": "P", "CCG": "P",
    "ACT": "T", "ACC": "T", "ACA": "T", "ACG": "T",
    "GCT": "A", "GCC": "A", "GCA": "A", "GCG": "A",
    "TAT": "Y", "TAC": "Y", "TAA": "*", "TAG": "*",
    "CAT": "H", "CAC": "H", "CAA": "Q", "CAG": "Q",
    "AAT": "N", "AAC": "N", "AAA": "K", "AAG": "K",
    "GAT": "D", "GAC": "D", "GAA": "E", "GAG": "E",
    "TGT": "C", "TGC": "C", "TGA": "*", "TGG": "W",
    "CGT": "R", "CGC": "R", "CGA": "R", "CGG": "R",
    "AGT": "S", "AGC": "S", "AGA": "R", "AGG": "R",
    "GGT": "G", "GGC": "G", "GGA": "G", "GGG": "G",
}
stop_codons = {"TAA", "TAG", "TGA"}

# The same short gene, sampled from two related species.
ancestor  = "ATGAAAGGTTTTCTGGAATATGCAGGCTAA"   # species A - the demos use this
species_b = "ATGAAAGGCTTCCTGGAGTATGCAAGCTAA"   # species B - your DIY challenges use this

print("species A:", ancestor)
print("species B:", species_b)


# our translate() from the functions notebooks
def translate(seq, frame=0):
    protein = ""
    for i in range(frame, len(seq) - 2, 3):
        codon = seq[i:i + 3]
        if codon in stop_codons:
            break
        protein = protein + codon_table[codon]
    return protein

# ...and some MESSY real-world sequences to break it on
messy = "ATGAANGGTTTT"        # an 'N' - the sequencer wasn't sure of this base
lower = "atgaaaggttttctg"     # lower-case letters

## Skill 1 — Read a traceback

When Python hits something it can't do, it **stops** and prints a **traceback**. Don't panic — read it **bottom-up**:

- the **last line** is the most important: the **error type** and a **message**
- above it, the line of code that triggered it

Let's cause one on purpose. `messy` contains an `N`, and `AAN` is not in `codon_table`.

In [ ]:
# DEMO - this WILL crash. Read the last line of the traceback.
translate(messy)          # 'AAN' is not a real codon -> KeyError

That last line — `KeyError: 'AAN'` — tells you exactly what went wrong: we asked the dictionary for a key that isn't there. **A `KeyError` is a fact, not a mystery.**

### 🛡️ DIY challenge

1. In a new cell, run `translate(lower)` — the lower-case sequence.
2. It crashes too. Read the last line: what **error type** is it, and which "codon" upset it?
3. In a comment, write one sentence explaining *why* lower-case breaks it. (Hint: what keys are in `codon_table`?)

In [ ]:
# TODO - your turn

## Skill 2 — `try` / `except`: catch it and carry on

A traceback **stops your whole program**. Often you'd rather **catch** the error and keep going:

```python
try:
    protein = translate(messy)
except KeyError:
    print("couldn't translate that one")
```

- the `try` block holds the risky code
- if it raises, Python jumps to the `except` block instead of crashing
- if it doesn't, the `except` block is skipped

In [ ]:
# DEMO - translate without crashing
try:
    protein = translate(messy)
    print("protein:", protein)
except KeyError:
    print("skipped: that sequence has a base I don't recognise")

print("...and the program keeps running.")

### 🛡️ DIY challenge

1. Wrap `translate(lower)` in a `try` / `except KeyError`.
2. In the `try`, print the protein; in the `except`, print a friendly "couldn't translate" message.
3. Add a `print("done")` *after* the whole thing to prove your program survived the error.

In [ ]:
# TODO - your turn

## Skill 3 — Catch the *right* error

`except:` on its own catches **everything** — including bugs you'd rather *see*. Always name the specific error you expect. Different problems raise different types:

| You did… | You get… |
|---|---|
| looked up a missing dict key | `KeyError` |
| `int("banana")` | `ValueError` |
| `seq[999]` past the end | `IndexError` |
| used a name that doesn't exist | `NameError` |

Catch `KeyError` and a *typo* in your own code still shows up loudly — which is what you want.

In [ ]:
# DEMO - a specific except leaves unrelated bugs visible
def safe_translate(seq):
    try:
        return translate(seq)
    except KeyError:
        return "?"          # unknown codon -> a clear placeholder

print("clean :", safe_translate(ancestor))
print("messy :", safe_translate(messy))

### 🛡️ DIY challenge

1. Call `safe_translate` on `ancestor`, `species_b`, `messy`, and `lower` in turn.
2. Which ones give a protein, and which give `"?"`?
3. `safe_translate` never crashes — but is `"?"` always the *right* answer? Jot one downside of silently returning `"?"`. (We'll fix messy data properly next.)

In [ ]:
# TODO - your turn

## Skill 4 — `raise` your own errors

Sometimes the best thing a function can do is **refuse** bad input — loudly, early, with a clear message. You `raise` an error yourself:

```python
if not all(base in "ATGC" for base in seq):
    raise ValueError(f"not DNA: {seq}")
```

This is exactly what `clean_dna` in Claudia's bee translator does. A good error message is a **gift** to whoever runs your program next (often future-you).

In [ ]:
# DEMO - reject anything that isn't clean DNA
def clean_dna(seq: str) -> str:
    """Upper-case a sequence and refuse anything that isn't A/T/G/C."""
    seq = seq.upper().strip()
    if not all(base in "ATGC" for base in seq):
        raise ValueError(f"that doesn't look like DNA: {seq}")
    return seq

print(clean_dna("atgaaaggt"))     # lower-case is fine - it gets cleaned up
clean_dna("ATG_XYZ")              # this raises ValueError on purpose

### 🛡️ DIY challenge

Write a checker `check_length(seq)` that guards a different problem — sequences whose length isn't a whole number of codons:

1. If `len(seq)` is **not** divisible by 3, `raise ValueError(...)` with a helpful message. (Hint: `len(seq) % 3 != 0`.)
2. Otherwise `return seq` unchanged.
3. Test it on `ancestor` (length 30 — fine) and on `"ATGAA"` (length 5 — should raise).

In [ ]:
# TODO - your turn

## 🔭 Explore further

- A `try` can have an `else` (ran only if no error) and a `finally` (ran **always**). Next notebook.
- Errors form a **family tree**: `KeyError` and `IndexError` are both a kind of `LookupError`; almost everything is an `Exception`. That's why `except Exception:` is such a wide net.
- You can attach a message to any error and read it later with `except ValueError as e: print(e)`.

## ✅ You've got it if you can…

- [ ] Read a traceback and name the **error type** from its last line
- [ ] Wrap risky code in `try` / `except` so it doesn't crash the program
- [ ] Catch a **specific** error type rather than a bare `except:`
- [ ] `raise` a `ValueError` with a clear message to reject bad input

**Feeling strong?** Open **`04_errors_advanced_concepts.ipynb`** for custom exception types and a batch translator that survives a whole folder of messy data.